# ML-03 — Frame Your Lane as an ML Task

This notebook frames the FlyRank content-refresh lane as a **scoring / ranking** problem. The goal is decision-support: prioritize pages for review rather than claim to predict Google's ranking algorithm.

The analysis uses the small anonymized FlyRank teaching dataset. No client names, URLs, titles, or private queries are used.

## 1. My lane as an ML task (type)

**Scoring / Ranking**

This is a ranking problem, not only a classification problem. The goal is to produce a continuous opportunity score that ranks pages from higher to lower priority for content review. A ranked queue is useful because a content team needs to decide which pages to investigate first.

In [ ]:
task_type = 'scoring / ranking'
reason = 'Produce an ordered review queue so the highest-opportunity pages can be investigated first.'
print(f'Task type: {task_type}')
print(f'Reason: {reason}')

## 2. Target or proxy

**Target (ideal, unmeasurable):**
Whether a page is genuinely under-capturing clicks relative to its true potential given its position and content quality.

**Proxy (measurable):**
`ctr_gap`: the difference between a page's actual CTR and the median CTR for pages in the same position tier. A strongly negative `ctr_gap` suggests the page is getting fewer clicks than comparable pages at a similar position.

**Proxy limitations:**
- Low CTR may reflect SERP feature competition, not title or content quality.
- Low CTR may reflect keyword intent mismatch.
- The proxy assumes position tier is an appropriate comparison group.
- CTR is an observed signal and is not a causal measure of content quality.

In [ ]:
from pathlib import Path
import pandas as pd

local_candidates = [
    Path.cwd() / 'data/raw/content_refresh_anonymized.csv',
    Path.cwd().parent.parent / 'data/raw/content_refresh_anonymized.csv',
]
data_path = next((p for p in local_candidates if p.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    print(f'Loaded local teaching dataset: {data_path}')
else:
    DATA_URL = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
    df = pd.read_csv(DATA_URL)
    print('Loaded the public anonymized FlyRank teaching dataset.')

required = {'ctr', 'position_tier', 'content_id', 'client_id', 'trend_direction'}
missing = required - set(df.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'

tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_gap'] = df['ctr'] - tier_median_ctr
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Proxy created: ctr_gap')
print(f'Declining label rate: {df.is_declining_label.mean():.3f}')

## 3. Success metric

**Primary: Precision@K (K = 20, K = 50)**

Of the top-K pages ranked by opportunity score, what fraction carry the `is_declining_label = 1` flag? A higher fraction means the ranked queue concentrates observed declining pages near the top.

**Baseline to beat:** a random ranking has Precision@K equal to the base rate of declining pages in the dataset. The learned scoring approach should exceed that baseline on held-out data before it is justified for operational use.

In [ ]:
def precision_at_k(frame, score_col, k):
    top_k = frame.nlargest(k, score_col)
    return top_k['is_declining_label'].mean()

# For framing only, use the inverse CTR gap as a transparent opportunity score.
# The later model notebook can learn a score from allowed features.
df['opportunity_score'] = -df['ctr_gap']
base_rate = df['is_declining_label'].mean()
p20 = precision_at_k(df, 'opportunity_score', 20)
p50 = precision_at_k(df, 'opportunity_score', 50)

print(f'Random baseline / declining base rate: {base_rate:.3f}')
print(f'Precision@20: {p20:.3f}')
print(f'Precision@50: {p50:.3f}')

## 4. The unit of analysis, as a real dataframe

**One row represents one anonymized content item/page.** The starter dataset contains page-level measurements aggregated over a trailing 90-day window. `content_id` identifies the pseudonymous page and `client_id` identifies its pseudonymous client group.

In [ ]:
preview_cols = [
    'content_id', 'client_id', 'search_volume', 'competition',
    'content_type', 'main_intent', 'word_count', 'impressions_90d',
    'clicks_90d', 'ctr', 'avg_position', 'position_tier',
    'trend_direction', 'trend_pct'
]
available = [c for c in preview_cols if c in df.columns]
display(df[available].head())
print('One row = one anonymized content item/page.')
print('Shape:', df.shape)
print('Unique content IDs:', df['content_id'].nunique())

## 5. Why ML beats a fixed rule here

A fixed rule such as `if CTR < X: review` is too rigid because CTR depends on several interacting factors, including search position, impression volume, content type, content age, update recency, search intent, and traffic context. The same CTR can mean different things for pages at different positions and volumes.

A rule also produces a coarse binary decision instead of a useful ordered queue. An ML model can learn combinations of measurable signals and produce a continuous score, then rank pages by that score.

A transparent hand-written rule remains a useful baseline. ML is justified only if it consistently improves Precision@K on held-out data. The resulting score should be treated as directional decision-support, not a causal claim.

In [ ]:
assert df['content_id'].is_unique, 'Expected one row per content item.'
assert df['is_declining_label'].isin([0, 1]).all()
assert df['trend_direction'].notna().all(), 'The label source should be present.'
print('Sanity checks passed.')
print('Candidate signals include position, traffic volume, content type, content age, update recency, and search intent.')

## Self-check

- [x] Every section above is filled — markdown reasoning and supporting code.
- [x] The notebook is designed to run top to bottom with the starter dataset available locally or through the public anonymized teaching dataset.
- [x] No client names, URLs, titles, or private queries are used in the analysis.
- [x] Claims use careful words: observed, measured, directional, proxy, and decision-support.
- [x] Committed under `work/notebooks/w02_ml_task_framing.ipynb`.

**Before portal submission:** run all cells once in Colab/Jupyter, confirm there are no errors, save the executed notebook, then submit the repository URL on the ML-03 assignment card.